In [1]:
# Solution Generator
#This Notebook generates actionable solutions based on top user issues extracted from reviews.

import pandas as pd
df=pd.read_csv("../data/BMW/clean_reviews.csv")

In [2]:
#Load Cluster and Issues

import pickle
with open("models/cluster_labels.pkl", "rb") as f:
    df["cluster"]= pickle.load(f)

In [3]:
#Define Issue Labels

issue_labels= {
    0: "Charging Issues",
    1: "Login Problems",
    2: "Connection Issues",
    3: "App Crashes",
    4: "Performance Issues"
}

In [4]:
#Filter Negative Reviews 

negative_reviews= df[df["score"]<=2]

In [5]:
#Helper Function

def get_reviews_by_cluster(cluster_id):
    return negative_reviews[negative_reviews["cluster"]==cluster_id]

In [6]:
#Solution Generator
import ollama
def generate_solution(cluster_id):
    reviews = get_reviews_by_cluster(cluster_id)
    if len(reviews)==0:
        return "No reviews found."
    reviews_sample = reviews.sample(min(10, len(reviews)))
    context = "\n".join(reviews_sample["content"].tolist())
    issue_name=issue_labels[cluster_id]

    prompt = f"""
You are a senior product manager.

Analyze the following user feedback and generate a concrete solution.

Issue:
{issue_name}

User Reviews:
{context}

Generate:
Problem Summary:
Root Cause:
Proposed Solution:
Key Features:
User Benefit:
Business Impact:
Implementation Complexity:
Priority:
"""

    try:
        response = ollama.chat(
            model="llama3",
            messages=[{"role": "user", "content": prompt}]
        )

        result = response["message"]["content"]
        return result

    except Exception as e:
        return f"LLM Error: {e}"

In [7]:
negative_reviews=negative_reviews.dropna(subset=["cluster"])

In [8]:
print(len(negative_reviews))

3


In [9]:
negative_reviews=df
print(negative_reviews["cluster"])

0              Vehicle Connectivity Issues
1          Login / Authentication Problems
2        Feature / Software Support Issues
3               App Functionality Problems
4        Charging / Remote Services Issues
                       ...                
10504                                  NaN
10505                                  NaN
10506                                  NaN
10507                                  NaN
10508                                  NaN
Name: cluster, Length: 10509, dtype: object


In [10]:
print(df["cluster"].unique())


['Vehicle Connectivity Issues' 'Login / Authentication Problems'
 'Feature / Software Support Issues' 'App Functionality Problems'
 'Charging / Remote Services Issues' nan]


In [12]:
print(generate_solution("Vehicle Connectivity Issues"))

KeyError: 'Vehicle Connectivity Issues'

In [ ]:
#Run for Top Issues
top_issues= negative_reviews["cluster"].value_counts().head(5)
    for cluster_id in top_issues.index:
        print("\n=================")
        print(issue_labels[cluster_id])
        print("=================\n")
        print(generate_solution(cluster_id))